# NoLLM Sentiment Analysis

Hugging Face sentiment inference and aspect aggregation. Full outputs retain historical rows; filtered export and summary use the agreed analytical scope.

**Execution:** Start Jupyter inside the project tree. Run the path setup first. Review configuration and existing outputs before executing.

In [ ]:
# Resolve the project from the notebook working directory, not a machine-specific path.
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if all((p / n).is_dir() for n in ("Python Code", "Data", "Streamlit"))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from inside the AEGEAN project tree.")
SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
CLEANING_DIR = PROJECT_ROOT / "Data" / "Output" / "Cleaning data"
HF_DIR = PROJECT_ROOT / "Data" / "Output" / "Hugging Face"
LLM_OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output" / "LLM"
for directory in (CLEANING_DIR, HF_DIR, LLM_OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


### Install libraries

In [ ]:
#%pip install -U pandas numpy torch transformers tqdm scikit-learn openpyxl

### Imports and configuration

In [ ]:
from pathlib import Path
import re
import json
import hashlib
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# ---------------------------------------------------
# PATHS
# ---------------------------------------------------

DATA_DIR = HF_DIR

INPUT_FILE = DATA_DIR / "df_final_features.csv"

OUTPUT_FILE = DATA_DIR / "df_final_absa.csv"

FILTERED_FILE = (
    DATA_DIR /
    "df_final_absa_no_first_class_no_long_haul.csv"
)

DIAGNOSTICS_FILE = DATA_DIR / "absa_diagnostics.csv"

SUMMARY_FILE = DATA_DIR / "absa_summary.csv"

# ---------------------------------------------------
# MODEL CONFIGURATION
# ---------------------------------------------------

MODEL_NAME = (
    "cardiffnlp/twitter-roberta-base-sentiment-latest"
)

DEVICE = (
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 32 if DEVICE == "cuda" else 16

MAX_LENGTH = 512

# Threshold for accepting a positive/negative
# sentence-level classification.
SENTIMENT_THRESHOLD = 0.55

# ---------------------------------------------------
# FEATURE CONFIGURATION
# ---------------------------------------------------

FEATURES = [
    "long_haul",
    "first_class",
    "cancellation",
    "delay",
    "boarding",
    "baggage",
    "economy_class",
    "business_class",
    "international",
    "domestic",
    "customer_service",
    "cabin_crew",
    "ground_staff",
    "complaint_resolution",
    "seat_comfort",
    "food_beverage",
    "cabin_cleanliness",
    "entertainment_wifi",
    "booking_experience",
    "check_in",
    "airport_experience",
    "value_for_money",
    "pricing_fees",
    "flight_disruption",
    "overall_experience"
]

LABELS = [
    "positive",
    "negative",
    "neutral",
    "mixed",
    "not_mentioned"
]

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Features:", len(FEATURES))

### Load and validate the dataset

In [ ]:
if "df_final_features" in globals():
    df = df_final_features.copy()
else:
    df = pd.read_csv(
        INPUT_FILE,
        encoding="utf-8-sig",
        low_memory=False
    )

required = [
    "record_key",
    "airline",
    "review_en",
    *FEATURES
]

missing = [
    col for col in required
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

if df["record_key"].isna().any():
    raise ValueError("Missing record_key values")

if not df["record_key"].is_unique:
    raise ValueError("record_key must be unique")

df["review_en"] = (
    df["review_en"]
    .fillna("")
    .astype(str)
)

for feature in FEATURES:
    df[feature] = pd.to_numeric(
        df[feature],
        errors="coerce"
    ).fillna(0).astype("int8")

    if not df[feature].isin([0, 1]).all():
        raise ValueError(
            f"Invalid binary values in {feature}"
        )

# Correct benchmark definitions
REGIONAL = {
    "Aegean Airlines",
    "Bangkok Air",
    "Porter Airlines",
    "Virgin Australia"
}

EUROPEAN = {
    "Aegean Airlines",
    "Air France",
    "Lufthansa",
    "Turkish Airlines"
}

df["regional"] = (
    df["airline"].isin(REGIONAL).astype("int8")
)

df["european"] = (
    df["airline"].isin(EUROPEAN).astype("int8")
)

print("Records:", len(df))
print("Non-empty reviews:",
      df["review_en"].str.strip().ne("").sum())

### Define aspect-specific vocabulary

In [ ]:
ASPECT_PATTERNS = {
    "long_haul": [
        r"\blong[\s-]?haul\b",
        r"\bintercontinental\b",
        r"\btransatlantic\b",
        r"\btranspacific\b"
    ],
    "first_class": [
        r"\bfirst[\s-]?class (?:seat|cabin|ticket|flight)\b",
        r"\bflew (?:in )?first class\b"
    ],
    "cancellation": [
        r"\bflight (?:was |got )?cancelled\b",
        r"\bflight (?:was |got )?canceled\b",
        r"\bcancellation\b",
        r"\bflight called off\b"
    ],
    "delay": [
        r"\bdelay(?:ed|s)?\b",
        r"\blate departure\b",
        r"\blate arrival\b",
        r"\bbehind schedule\b"
    ],
    "boarding": [
        r"\bboarding\b",
        r"\bboarded\b",
        r"\bboarding queue\b",
        r"\bgate agents?\b"
    ],
    "baggage": [
        r"\bbaggage\b",
        r"\bluggage\b",
        r"\bsuitcase\b",
        r"\bcarry[\s-]?on\b",
        r"\bchecked bags?\b",
        r"\boverhead bins?\b"
    ],
    "economy_class": [
        r"\beconomy[\s-]?class\b",
        r"\beconomy seat\b",
        r"\bflew economy\b",
        r"\btravelled in coach\b",
        r"\btraveled in coach\b"
    ],
    "business_class": [
        r"\bbusiness[\s-]?class\b",
        r"\bbusiness cabin\b",
        r"\bbusiness seat\b"
    ],
    "international": [
        r"\binternational flight\b",
        r"\binternational route\b",
        r"\bcross[\s-]?border flight\b"
    ],
    "domestic": [
        r"\bdomestic flight\b",
        r"\bdomestic route\b",
        r"\binternal flight\b"
    ],
    "customer_service": [
        r"\bcustomer (?:service|support|care)\b",
        r"\bcall cent(?:er|re)\b",
        r"\bhelpdesk\b",
        r"\bhotline\b",
        r"\bcontact cent(?:er|re)\b"
    ],
    "cabin_crew": [
        r"\bcabin crew\b",
        r"\bflight attendants?\b",
        r"\bstewards?\b",
        r"\bstewardesses\b",
        r"\bonboard staff\b"
    ],
    "ground_staff": [
        r"\bground staff\b",
        r"\bground crew\b",
        r"\bcheck[\s-]?in agents?\b",
        r"\bgate agents?\b",
        r"\bairport staff\b"
    ],
    "complaint_resolution": [
        r"\bcomplaints?\b",
        r"\brefunds?\b",
        r"\bcompensation\b",
        r"\breimbursement\b",
        r"\bclaim (?:was |is )?(?:rejected|denied|resolved)\b",
        r"\bunresolved\b"
    ],
    "seat_comfort": [
        r"\bseat(?:s)?\b",
        r"\blegroom\b",
        r"\brecline\b",
        r"\bseat pitch\b",
        r"\bcramped\b"
    ],
    "food_beverage": [
        r"\bfood\b",
        r"\bmeals?\b",
        r"\bsnacks?\b",
        r"\bdrinks?\b",
        r"\bbeverages?\b",
        r"\bcatering\b",
        r"\bbreakfast\b",
        r"\bdinner\b"
    ],
    "cabin_cleanliness": [
        r"\bcleanliness\b",
        r"\bclean cabin\b",
        r"\bdirty cabin\b",
        r"\bdirty seats?\b",
        r"\bfilthy\b",
        r"\btoilets?\b",
        r"\blavator(?:y|ies)\b"
    ],
    "entertainment_wifi": [
        r"\bwi[\s-]?fi\b",
        r"\bentertainment\b",
        r"\bife\b",
        r"\bmovies?\b",
        r"\bcharging ports?\b",
        r"\bpower outlets?\b"
    ],
    "booking_experience": [
        r"\bbooking\b",
        r"\breservation\b",
        r"\bwebsite\b",
        r"\bmobile app\b",
        r"\bpayment process\b",
        r"\brebooking\b"
    ],
    "check_in": [
        r"\bcheck[\s-]?in\b",
        r"\bcheck[\s-]?in counters?\b",
        r"\bself[\s-]?service kiosks?\b"
    ],
    "airport_experience": [
        r"\bairport lounge\b",
        r"\blounge\b",
        r"\bterminal\b",
        r"\bairport facilities\b",
        r"\bairport transfer\b"
    ],
    "value_for_money": [
        r"\bvalue for money\b",
        r"\bworth (?:the|my|every) money\b",
        r"\bworth the price\b",
        r"\bgood value\b",
        r"\bnot worth\b",
        r"\boverpriced\b"
    ],
    "pricing_fees": [
        r"\bprice\b",
        r"\bpricing\b",
        r"\bexpensive\b",
        r"\bcheap\b",
        r"\bfees?\b",
        r"\bcharges?\b",
        r"\bfare\b",
        r"\bextra cost\b"
    ],
    "flight_disruption": [
        r"\bmissed connection\b",
        r"\bmissed my connecting flight\b",
        r"\bdiverted\b",
        r"\bdiversion\b",
        r"\bstranded\b",
        r"\brebooked\b",
        r"\bdisruption\b"
    ],
    "overall_experience": [
        r"\boverall experience\b",
        r"\bwhole experience\b",
        r"\bentire experience\b",
        r"\bexperience with (?:this|the) airline\b",
        r"\bwould (?:not )?fly (?:with )?(?:them|again)\b",
        r"\bnever again\b",
        r"\bhighly recommend\b"
    ]
}

COMPILED_PATTERNS = {
    feature: re.compile(
        "|".join(patterns),
        flags=re.IGNORECASE
    )
    for feature, patterns in ASPECT_PATTERNS.items()
}

### Prepare review text and extract relevant segments

In [ ]:
SENTENCE_SPLIT = re.compile(
    r"(?<=[.!?])\s+|\n+"
)

CONTRAST_SPLIT = re.compile(
    r"\s*[,;]?\s+\b(?:but|however|although|whereas)\b\s*",
    flags=re.IGNORECASE
)

def split_review(text):
    """Return or process split review."""
    if not isinstance(text, str) or not text.strip():
        return []

    sentences = SENTENCE_SPLIT.split(text.strip())

    segments = []

    for sentence in sentences:
        clauses = CONTRAST_SPLIT.split(sentence)

        for clause in clauses:
            clause = clause.strip(" ,;:-")

            if clause:
                segments.append(clause)

    return segments


# Prepare each distinct review text once
unique_texts = df["review_en"].drop_duplicates()

text_segments = {
    text: split_review(text)
    for text in tqdm(
        unique_texts,
        desc="Segmenting reviews"
    )
}

# Identify distinct segments for model inference
unique_segments = list({
    segment
    for segments in text_segments.values()
    for segment in segments
})

print("Unique review texts:", len(unique_texts))
print("Unique text segments:", len(unique_segments))

### Load the Hugging Face sentiment model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

model.to(DEVICE)
model.eval()

print("Model loaded successfully")
print("Device:", DEVICE)

### Run sentiment inference

In [ ]:
def predict_sentiment_batches(
    texts,
    batch_size=BATCH_SIZE
):
    """Return or process predict sentiment batches."""
    results = {}

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Hugging Face sentiment inference"
    ):

        batch = texts[start:start + batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoded = {
            k: v.to(DEVICE)
            for k, v in encoded.items()
        }

        with torch.inference_mode():
            logits = model(**encoded).logits

            probabilities = (
                torch.softmax(logits, dim=-1)
                .cpu()
                .numpy()
            )

        for text, probs in zip(batch, probabilities):

            score_map = {
                model.config.id2label[i].lower(): float(p)
                for i, p in enumerate(probs)
            }

            # Expected model labels:
            # negative, neutral, positive
            results[text] = score_map

    return results


segment_scores = predict_sentiment_batches(
    unique_segments
)

print(
    f"Successfully classified "
    f"{len(segment_scores):,} unique segments"
)

### Aggregate sentiment into five labels

In [ ]:
def aggregate_sentiment(
    segments,
    scores,
    threshold=SENTIMENT_THRESHOLD
):
    """Return or process aggregate sentiment."""
    if not segments:
        return "not_mentioned", np.nan

    probabilities = [
        scores[s]
        for s in segments
        if s in scores
    ]

    if not probabilities:
        return "not_mentioned", np.nan

    positive = [
        p["positive"]
        for p in probabilities
    ]

    negative = [
        p["negative"]
        for p in probabilities
    ]

    neutral = [
        p["neutral"]
        for p in probabilities
    ]

    has_positive = any(
        p >= threshold
        for p in positive
    )

    has_negative = any(
        p >= threshold
        for p in negative
    )

    if has_positive and has_negative:
        return "mixed", float(
            min(max(positive), max(negative))
        )

    if has_positive:
        return "positive", float(max(positive))

    if has_negative:
        return "negative", float(max(negative))

    mean_probs = {
        "positive": np.mean(positive),
        "negative": np.mean(negative),
        "neutral": np.mean(neutral)
    }

    label = max(
        mean_probs,
        key=mean_probs.get
    )

    # Conservative fallback when the evidence
    # is not sufficiently strong.
    if label != "neutral":
        return "neutral", float(mean_probs[label])

    return "neutral", float(mean_probs["neutral"])

### Calculate overall review sentiment

In [ ]:
overall_cache = {}

for text, segments in tqdm(
    text_segments.items(),
    desc="Overall review sentiment"
):
    overall_cache[text] = aggregate_sentiment(
        segments,
        segment_scores
    )

overall_results = df["review_en"].map(
    overall_cache
)

df["overall_sentiment"] = overall_results.map(
    lambda x: x[0]
)

df["overall_sentiment_score"] = overall_results.map(
    lambda x: x[1]
)

print(
    df["overall_sentiment"]
    .value_counts(dropna=False)
)

### Calculate aspect-based sentiment for all 25 features

In [ ]:
def classify_aspect_sentiment(
    segments,
    feature
):
    """Return or process classify aspect sentiment."""
    pattern = COMPILED_PATTERNS[feature]

    matched = [
        segment
        for segment in segments
        if pattern.search(segment)
    ]

    if not matched:
        return (
            "not_mentioned",
            np.nan,
            ""
        )

    label, score = aggregate_sentiment(
        matched,
        segment_scores
    )

    evidence = " | ".join(matched[:5])

    return label, score, evidence


# Cache classification by unique text and feature
aspect_cache = {}

for text, segments in tqdm(
    text_segments.items(),
    desc="Aspect sentiment"
):
    aspect_cache[text] = {
        feature: classify_aspect_sentiment(
            segments,
            feature
        )
        for feature in FEATURES
    }


# Generate 25 sentiment columns
# and associated diagnostics
diagnostic_rows = []

for feature in tqdm(
    FEATURES,
    desc="Building ABSA columns"
):

    sentiments = []
    scores = []
    evidence_list = []

    for text, binary_value in zip(
        df["review_en"],
        df[feature]
    ):

        if binary_value == 0:
            label, score, evidence = (
                "not_mentioned",
                np.nan,
                ""
            )
        else:
            label, score, evidence = (
                aspect_cache[text][feature]
            )

        sentiments.append(label)
        scores.append(score)
        evidence_list.append(evidence)

    df[f"{feature}_sentiment"] = sentiments

    # Keep scores and evidence in diagnostics,
    # rather than expanding the analytical dataset
    diagnostic_rows.append(
        pd.DataFrame({
            "record_key": df["record_key"].values,
            "feature": feature,
            "binary_classification": df[feature].values,
            "sentiment": sentiments,
            "sentiment_score": scores,
            "evidence": evidence_list
        })
    )

df_absa_diagnostics = pd.concat(
    diagnostic_rows,
    ignore_index=True
)

print("ABSA completed")
print("Dataset shape:", df.shape)

### Validate the results

In [ ]:
SENTIMENT_COLUMNS = [
    f"{feature}_sentiment"
    for feature in FEATURES
]

SENTIMENT_COLUMNS.append(
    "overall_sentiment"
)

# Check labels
for col in SENTIMENT_COLUMNS:

    invalid = ~df[col].isin(LABELS)

    if invalid.any():
        raise ValueError(
            f"Invalid sentiment labels in {col}"
        )

# Check row preservation
assert len(df) == len(df_final_features) if (
    "df_final_features" in globals()
) else len(df) > 0

# Check identifier uniqueness
assert df["record_key"].is_unique

# Check that features remain binary
for feature in FEATURES:
    assert df[feature].isin([0, 1]).all()

# No sentiment assigned to absent features
for feature in FEATURES:

    invalid = (
        (df[feature] == 0)
        & (
            df[f"{feature}_sentiment"]
            != "not_mentioned"
        )
    )

    assert not invalid.any(), feature

print("All structural validation checks passed")

### Generate sentiment summaries

In [ ]:
# Restrict business summaries without discarding the full inference dataset.
analysis_df = df.loc[pd.to_datetime(df["review_date"], errors="coerce", utc=True).ge(pd.Timestamp("2024-01-01", tz="UTC")) & df["long_haul"].ne(1) & df["first_class"].ne(1)].copy()
if analysis_df.empty:
    raise ValueError("No eligible reviews for analytical summaries.")
# Overall review sentiment
overall_summary = (
    analysis_df["overall_sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="reviews")
)

overall_summary["percentage"] = (
    overall_summary["reviews"]
    / len(analysis_df) * 100
)

display(overall_summary)


# Aspect sentiment distribution
summary_rows = []

for feature in FEATURES:

    counts = (
        analysis_df[f"{feature}_sentiment"]
        .value_counts()
    )

    for label in LABELS:
        summary_rows.append({
            "feature": feature,
            "sentiment": label,
            "reviews": int(counts.get(label, 0)),
            "percentage_all_reviews": (
                counts.get(label, 0) / len(analysis_df) * 100
            )
        })

df_absa_summary = pd.DataFrame(
    summary_rows
)

display(df_absa_summary)

### Compare airlines

In [ ]:
airline_sentiment = pd.crosstab(
    analysis_df["airline"],
    analysis_df["overall_sentiment"]
)

airline_sentiment_pct = (
    airline_sentiment
    .div(
        airline_sentiment.sum(axis=1),
        axis=0
    )
    .mul(100)
    .round(2)
)

display(airline_sentiment_pct)

### compare aspect sentiment across airlines

In [ ]:
def compare_aspect(feature):

    """Return or process compare aspect."""
    if feature not in FEATURES:
        raise ValueError("Unknown feature")

    subset = analysis_df[
        analysis_df[f"{feature}_sentiment"]
        != "not_mentioned"
    ]

    result = pd.crosstab(
        subset["airline"],
        subset[f"{feature}_sentiment"],
        normalize="index"
    ).mul(100).round(2)

    return result


display(
    compare_aspect("customer_service")
)

display(
    compare_aspect("cabin_crew")
)

display(
    compare_aspect("seat_comfort")
)

### Save both final datasets and diagnostics

In [ ]:
# Full analytical dataset
df_final_absa = df.copy()

# Exclude identified first-class and long-haul reviews
df_final_absa_filtered = df_final_absa[
    df_final_absa["first_class"].ne(1)
    & df_final_absa["long_haul"].ne(1)
    & pd.to_datetime(df_final_absa["review_date"], errors="coerce", utc=True).ge(pd.Timestamp("2024-01-01", tz="UTC"))
].copy()

# Save outputs
df_final_absa.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

df_final_absa_filtered.to_csv(
    FILTERED_FILE,
    index=False,
    encoding="utf-8-sig"
)

df_absa_diagnostics.to_csv(
    DIAGNOSTICS_FILE,
    index=False,
    encoding="utf-8-sig"
)

df_absa_summary.to_csv(
    SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig"
)

airline_sentiment_pct.to_csv(
    DATA_DIR / "absa_airline_sentiment_summary.csv",
    encoding="utf-8-sig"
)

print("Files saved successfully")
print("Full dataset:", len(df_final_absa))
print("Filtered dataset:", len(df_final_absa_filtered))